# MFCC Output Analysis

Quick sanity checks for the extracted MFCC feature CSV.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path.cwd().parents[0]
csv_path = repo_root / "extracted_features" / "mfcc" / "mfcc_features.csv"
csv_path

WindowsPath('f:/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv')

In [11]:
df = pd.read_csv(csv_path)
df.shape

(7532, 1157)

In [12]:
df.head()

,path,session,method,gender,emotion,n_annotators,agreement,duration_s,mfcc13_mfcc_frames,mfcc13_mfcc_mean,...,mfcc40_d2_c38_mean,mfcc40_d2_c38_std,mfcc40_d2_c38_min,mfcc40_d2_c38_max,mfcc40_d2_c38_median,mfcc40_d2_c39_mean,mfcc40_d2_c39_std,mfcc40_d2_c39_min,mfcc40_d2_c39_max,mfcc40_d2_c39_median
0,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,3,2.070000,130.0,-25.572800,...,-0.006784,0.467840,-0.933521,1.694955,-0.030211,-0.024009,0.498571,-1.077097,1.030934,-0.048602
1,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,fru,3,2,1.502438,94.0,-27.364491,...,0.025404,0.453846,-0.837526,1.320832,-0.024747,-0.007647,0.576057,-1.555333,1.227596,0.049068
2,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,sur,3,2,1.970000,124.0,-24.889765,...,0.030572,0.727588,-2.299917,2.850854,-0.015772,0.015152,0.648481,-1.382465,2.908650,0.049970
3,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,2,2.180000,137.0,-23.416510,...,0.063818,1.004474,-3.346304,2.417407,-0.067181,0.016259,0.765166,-2.333637,2.718325,0.023806
4,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,ang,3,2,2.970000,186.0,-21.269238,...,-0.017926,0.931527,-2.260671,3.068222,-0.092336,-0.018099,0.871944,-3.015595,2.134903,-0.034257


In [13]:
# Basic column counts
numeric_cols = df.select_dtypes(include=[np.number]).columns
non_numeric_cols = df.columns.difference(numeric_cols)

len(df.columns), len(numeric_cols), len(non_numeric_cols)

(1157, 1153, 4)

In [14]:
# Missing values summary
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(20)

Series([], dtype: int64)

In [15]:
# Overall numeric ranges
global_min = df[numeric_cols].min().min()
global_max = df[numeric_cols].max().max()
global_min, global_max

(np.float64(-707.4547119140625), np.float64(2134.0))

In [16]:
# Per-feature min/max/range (top 20 widest ranges)
ranges = df[numeric_cols].agg(["min", "max"]).T
ranges["range"] = ranges["max"] - ranges["min"]
ranges.sort_values("range", ascending=False).head(20)

,min,max,range
mfcc13_mfcc_frames,37.000000,2134.000000,2097.000000
mfcc13_d1_frames,37.000000,2134.000000,2097.000000
mfcc40_d1_frames,37.000000,2134.000000,2097.000000
mfcc40_d2_frames,37.000000,2134.000000,2097.000000
mfcc13_d2_frames,37.000000,2134.000000,2097.000000
mfcc20_d1_frames,37.000000,2134.000000,2097.000000
mfcc20_mfcc_frames,37.000000,2134.000000,2097.000000
mfcc40_mfcc_frames,37.000000,2134.000000,2097.000000
mfcc20_d2_frames,37.000000,2134.000000,2097.000000
mfcc13_mfcc_c00_max,-584.381042,96.266235,680.647278


In [17]:
# MFCC feature group sizes
groups = {
    "mfcc13": df.filter(regex=r"^mfcc13").shape[1],
    "mfcc20": df.filter(regex=r"^mfcc20").shape[1],
    "mfcc40": df.filter(regex=r"^mfcc40").shape[1],
}
groups

{'mfcc13': 213, 'mfcc20': 318, 'mfcc40': 618}

In [18]:
# Quick stats for duration and frame counts
df[["duration_s", "mfcc13_mfcc_frames", "mfcc20_mfcc_frames", "mfcc40_mfcc_frames"]].describe()

,duration_s,mfcc13_mfcc_frames,mfcc20_mfcc_frames,mfcc40_mfcc_frames
count,7532.000000,7532.000000,7532.000000,7532.000000
mean,4.558042,285.367631,285.367631,285.367631
std,3.157435,197.343038,197.343038,197.343038
min,0.584937,37.000000,37.000000,37.000000
25%,2.341172,147.000000,147.000000,147.000000
50%,3.610000,226.000000,226.000000,226.000000
75%,5.850000,366.000000,366.000000,366.000000
max,34.138750,2134.000000,2134.000000,2134.000000


In [19]:
# Emotion distribution (sanity check after filtering)
df["emotion"].value_counts()

emotion
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
sur     107
fea      40
oth       3
dis       2
Name: count, dtype: int64